In [3]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path(".")   # 如果你的 tree 输出就是在当前目录，这里保持 "."
TASKS = ["sib200", "xnli"]
SPLITS = ["train", "test","validation"]

def file_to_accuracy(path: Path):
    # 读取 JSON Lines：每行一个 JSON 对象
    df = pd.read_json(path, lines=True)

    lang = path.stem  # ar/de/en/sw/vi/zh

    # orig_pred 是个 dict，比如 {"en": 6}；取出对应语言的预测
    # 同时防御性地转成数值（有时可能读成 object）
    pred = df["orig_pred"].apply(lambda d: d.get(lang) if isinstance(d, dict) else None)
    label = pd.to_numeric(df["label"], errors="coerce")
    pred = pd.to_numeric(pred, errors="coerce")

    ok = (label == pred)
    n = int(ok.notna().sum())
    acc = float(ok.mean()) if n > 0 else float("nan")
    return lang, n, acc

rows = []
for task in TASKS:
    for split in SPLITS:
        dirp = DATA_ROOT / task / split
        for path in sorted(dirp.glob("*.jsonl")):
            lang, n, acc = file_to_accuracy(path)
            rows.append({
                "task": task,
                "split": split,
                "lang": lang,
                "n": n,
                "accuracy": acc,
                "file": str(path),
            })

res = pd.DataFrame(rows).sort_values(["task", "split", "lang"]).reset_index(drop=True)
res


,task,split,lang,n,accuracy,file
0,sib200,test,en,9,0.888889,sib200/test/en.jsonl
1,sib200,test,sw,9,0.555556,sib200/test/sw.jsonl
2,sib200,test,zh,9,0.222222,sib200/test/zh.jsonl
3,sib200,train,en,52,0.750000,sib200/train/en.jsonl
4,sib200,train,sw,50,0.420000,sib200/train/sw.jsonl
5,sib200,train,zh,50,0.760000,sib200/train/zh.jsonl
6,sib200,validation,en,9,0.888889,sib200/validation/en.jsonl
7,sib200,validation,sw,9,0.555556,sib200/validation/sw.jsonl
8,sib200,validation,zh,9,0.888889,sib200/validation/zh.jsonl
9,xnli,test,en,10,0.900000,xnli/test/en.jsonl


# target alignment with en

In [1]:
import json
from pathlib import Path
import pandas as pd

DATA_ROOT = Path("../data_qwen_pred")  # adjust if your notebook cwd is different
DATASETS = ["xnli", "sib200"]
SPLITS = ["train", "validation", "test"]
LANGS = ["ar", "de", "ru", "sw", "vi", "zh"]  # non-English

def read_jsonl(path: Path):
    items = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            items.append(json.loads(line))
    return items

def get_lang_int(obj, field, lang):
    d = obj.get(field, {})
    if not isinstance(d, dict) or lang not in d:
        return None
    v = d[lang]
    try:
        return int(v)
    except Exception:
        return None

rows = []

for ds in DATASETS:
    for sp in SPLITS:
        split_dir = DATA_ROOT / ds / sp
        en_path = split_dir / "en.jsonl"
        if not en_path.exists():
            continue

        en_items = read_jsonl(en_path)
        # map: index -> en_target
        en_target = {}
        for o in en_items:
            idx = o.get("index")
            t = get_lang_int(o, "target_pred", "en")
            if idx is not None and t is not None:
                en_target[idx] = t

        # per-language alignment
        for lang in LANGS:
            p = split_dir / f"{lang}.jsonl"
            if not p.exists():
                continue
            items = read_jsonl(p)

            total = 0
            aligned = 0
            missing_en = 0
            missing_target = 0

            for o in items:
                idx = o.get("index")
                lt = get_lang_int(o, "target_pred", lang)
                et = en_target.get(idx, None)

                if et is None:
                    missing_en += 1
                    continue
                if lt is None:
                    missing_target += 1
                    continue

                total += 1
                if lt == et:
                    aligned += 1

            rows.append({
                "dataset": ds,
                "split": sp,
                "lang": lang,
                "aligned": aligned,
                "total_compared": total,
                "align_rate": (aligned / total) if total > 0 else None,
                "missing_en": missing_en,
                "missing_lang_target": missing_target,
                "file": str(p)
            })

df = pd.DataFrame(rows)

# Show per-language alignment rates
df.sort_values(["dataset", "split", "lang"]).reset_index(drop=True)



KeyError: 'dataset'